In [25]:
!python -m ipykernel install --user --name=venv --display-name="Python (venv)"

Installed kernelspec venv in C:\Users\brunab.UNESC\AppData\Roaming\jupyter\kernels\venv


In [26]:
!pip install geopandas folium numpy ipywidgets

In [27]:
import os
import folium
import json
from folium.plugins import LocateControl
print(folium.__version__)

import geopandas as gpd
from folium import plugins
from folium.plugins import Geocoder
from folium.plugins import FloatImage
import base64
from folium.plugins import FloatImage
#plguin MiniMap
from folium.plugins import MiniMap
#abrir o mapa interativo diretamente no browser
import webbrowser
import numpy as np
from shapely.geometry import Point, Polygon
from ipywidgets import interact, widgets

0.20.0


CRIACAO DO MAPA INTERATIVO

In [28]:
#CRIACAO DO MAPA INTERATIVO
#mapaint = folium.Map(width=970, height=800,location=[-28.46466958, -48.79780130], zoom_start=4)
mapaint = folium.Map(location=[-28.75020792, -49.11252282], zoom_start=16,control_scale=True, tiles=None)

Localização em tempo real

In [29]:
#folium.plugins.LocateControl().add_to(mapaint)

# If you want get the user device position after load the map, set auto_start=True
folium.plugins.LocateControl(auto_start=True).add_to(mapaint)

In [30]:
#no Leaflet Provider Demo eu posso escolher uma basemap que quero, até mais de uma, basta acrescentar aqui com o comando folium.TileLayer
folium.TileLayer(tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
                 #attr = 'Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',
                 attr = 'Autoria: Bruna B. da Rocha (2026); Fontes: BaseMap: Esri.WorldImagery',
                 name = "Base Map: Esri.WorldImagery").add_to(mapaint)

#no Leaflet Provider Demo eu posso escolher uma basemap que quero, até mais de uma, basta acrescentar aqui com o comando folium.TileLayer
folium.TileLayer(tiles = 'https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png',
                 #attr = 'Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',
                 attr = 'Autoria: Bruna B. da Rocha (2026); Base Map: CartoDB.DarkMatter',
                 name = "Base Map: CartoDB.DarkMatter").add_to(mapaint)

#no Leaflet Provider Demo eu posso escolher uma basemap que quero, até mais de uma, basta acrescentar aqui com o comando folium.TileLayer
folium.TileLayer(tiles = 'https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
                 #attr = 'Tiles &copy; Esri &mdash; Source: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',
                 attr = 'Autoria: Bruna B. da Rocha (2026); Base Map: Esri.WorldStreetMap',
                 name = "Base Map: Esri.WorldStreetMap").add_to(mapaint)



In [31]:
folium.LayerControl().add_to(mapaint)

#aqui eu configuro o minimapa como eu quero, sua posição / função "toggle_display" que possibilita a minimização do minimap ao clicar nela
#se eu colocar "auto_toggle_display" ao invés de toggle_display, o minimapa desaparece automaticamente, sem eu precisar clicar na flechinha
#com a função "zoom_level_fixed" eu fixo o minimapa na área que eu quiser
MiniMap(tiles_layer='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png"', position = "bottomright", toggle_display = True, zoom_level_fixed = 9).add_to(mapaint)

#adicionar um título e configurando a formatação dele
#titulo = "Mapa interativo"
#titulo_html = """<h1 align = "center", style="font-family: verdana; font-size:14px;
#font-wight:bold; background-color: black; color: white; padding:10px; text-transfore: uppercase;">{}</h1>""".format(titulo)
#mapaint.get_root().html.add_child(folium.Element(titulo_html))

#aqui eu adiciono uma caixa de busca no layer, configurando sua posição e o marcador (se deve ou não aparecer no local que eu buscar)
Geocoder(collapsed = True, position = "topleft", add_marker = True).add_to(mapaint)

mapaint.add_child(folium.LatLngPopup())

nome_do_arquivo = "Norte.png"

#aqui o "rb" informa par ao base64 que a imagem deve ser lida como binária / depois é preciso transformar a imagem em utf-8
#peço para o base lê o binário e converte para utf-8
with open(nome_do_arquivo, "rb") as arquivo_imagem:
    string_imagem = base64.b64encode(arquivo_imagem.read()).decode("utf-8")

#print(string_imagem)

#adicionando a imagem em formato de string
#preciso informar que essa imagem é em formato png e foi convertida para o base64
FloatImage("data:image/png; base64, {}".format(string_imagem), bottom=5.5, left=0.5).add_to(mapaint)

# Adicionar o controle de desenho (Draw) ao mapa
#draw = plugins.Draw(export=False)
#FONTE: https://gis.stackexchange.com/questions/445450/exporting-geojson-with-defined-properties-using-folium-draw-plugin
draw = plugins.Draw(export=True,
     filename='arquivo_gerado.geojson',
     position = 'topleft').add_to(mapaint)

mapObjectInHTML = mapaint.get_name()

mapaint.get_root().html.add_child(folium.Element("""
<script type="text/javascript">
  $(document).ready(function(){           
    var x = 0;
    
    {map}.on("draw:created", function(e){    
        var layer = e.layer;
            feature = layer.feature = layer.feature || {}; 
            
        var camada = prompt("Insira o nome da camada", "nome");
        var valor = prompt("Insira um valor", "valor");
        var id = x++;

        feature.type = feature.type || "Feature";
        var props = feature.properties = feature.properties || {};
        props.Id = id;
        props.Camada = camada;
        props.Valor = valor;
        drawnItems.addLayer(layer);
      });    
    });    
</script>
""".replace('{map}', mapObjectInHTML)))
draw.add_to(mapaint)

# Adicionar plugins extras
plugins.Fullscreen(position='topleft').add_to(mapaint)  # Plugin de tela cheia
#plugins.MeasureControl(position='bottomleft', left = 3,primary_length_unit='Km').add_to(mapaint)  # Plugin de medição
plugins.MousePosition().add_to(mapaint)  # Plugin para mostrar as coordenadas ao passar o mouse

mapaint 

In [32]:
#aqui eu salvo o mapa como html
mapaint.save("Interactive_map_Export.html")

# Abrindo o mapa com marcadores no navegador padrão
webbrowser.open('Interactive_map_Export.html')

True